# 授業方略実験ノートブック

最終更新: 2026-07-20 13:43:41 JST

このノートブックでは、教師AIが授業手法を考える前段階を確認します。生徒AIの発話を作り、伝達AIが観察し、教師AI用のコンテキストを作成して、次の授業方略を選びます。

## 1. GitHubから最新版を取得

Colabでは最初にこのセルを実行します。まだcloneしていない場合はcloneし、すでにclone済みの場合は最新版をpullします。

In [ ]:
from pathlib import Path

REPO_URL = "https://github.com/Hiromu-0219/student-ai-test.git"
PROJECT_DIR = Path("/content/student-ai")

if not PROJECT_DIR.exists():
    !git clone {REPO_URL} {PROJECT_DIR}

%cd /content/student-ai
!git status --short --branch
!git pull
!git log -1 --oneline
!grep -n "class ClassManager" src/class_manager.py
!grep -n "summarize_classroom" src/observer/trait_classifier.py
!grep -n "classroom_observation" src/teacher/context_builder.py
!grep -n "TeacherBeliefManager" src/teacher/belief_manager.py
!grep -n "RuleBasedInterventionPlanner" src/teacher/intervention_planner.py
!grep -n "RuleBasedTeacherUtteranceBuilder" src/teacher/utterance_builder.py
!grep -n "LLMTeacherUtteranceBuilder" src/teacher/utterance_builder.py
!grep -n "RuleBasedLessonPlanner" src/teacher/lesson_planner.py
!grep -n "LessonSessionRunner" src/teacher/lesson_session_runner.py

## 2. 依存関係をインストール

In [ ]:
!pip install -q -r requirements.txt

## 3. Experiment settings

Default mode runs only the core experiment. Heavy validation cells are skipped unless their `RUN_*` flag is set to `True`.

In [ ]:
STUDENT_ID = "S002"
CLASS_ID = "class_3_basic"  # class_3_basic / class_5_mixed / class_10_mixed / class_20_mixed

# Core LLM switches.
USE_LLM_STUDENT_UTTERANCES = True
USE_LLM_COMMUNICATION_AI = True
USE_LLM_TEACHER_UTTERANCES = True

# Heavy optional cells. Keep False for quick checks.
RUN_MULTI_LESSON_VALIDATION = False
RUN_SCENARIO_COMPARISON = False
RUN_NEXT_TURN_SIMULATION = False
RUN_AUTO_LESSON_SESSION = False
RUN_LEGACY_SINGLE_STUDENT_CONTEXT = False
SAVE_FULL_ARTIFACTS = False

# Fast bridge check model. For final 4B validation, use: "Qwen/Qwen3-4B"
SHARED_LLM_MODEL_ID = "Qwen/Qwen2.5-1.5B-Instruct"
SHARED_LLM_LOAD_IN_4BIT = True

USE_MOCK_MODEL = not USE_LLM_STUDENT_UTTERANCES
TEACHER_MESSAGE = "x + 3 = 8 \u3092\u89e3\u3044\u3066\u307f\u307e\u3057\u3087\u3046\u3002+3\u3092\u53cd\u5bfe\u5074\u3078\u79fb\u3059\u3068\u3001\u7b26\u53f7\u306f\u3069\u3046\u306a\u308a\u307e\u3059\u304b\u3002"

print("student_id:", STUDENT_ID)
print("class_id:", CLASS_ID)
print("use_llm_student_utterances:", USE_LLM_STUDENT_UTTERANCES)
print("use_llm_communication_ai:", USE_LLM_COMMUNICATION_AI)
print("use_llm_teacher_utterances:", USE_LLM_TEACHER_UTTERANCES)
print("shared_llm_model_id:", SHARED_LLM_MODEL_ID)
print("shared_llm_load_in_4bit:", SHARED_LLM_LOAD_IN_4BIT)
print("run_multi_lesson_validation:", RUN_MULTI_LESSON_VALIDATION)
print("run_scenario_comparison:", RUN_SCENARIO_COMPARISON)
print("run_next_turn_simulation:", RUN_NEXT_TURN_SIMULATION)
print("run_auto_lesson_session:", RUN_AUTO_LESSON_SESSION)
print("teacher_message:", TEACHER_MESSAGE)

## 4. 生徒AIの発話を生成

ここでは授業中の対話として生徒発話を生成します。実験を繰り返しやすくするため、標準では知識状態を更新しません。

In [ ]:
import importlib

from src.config import GenerationConfig, ModelLoadConfig
from src.model_loader import LocalLLM
from src.student_ai import StudentAISimulator


def ensure_bitsandbytes_for_shared_llm():
    try:
        from transformers.utils import is_bitsandbytes_available
        if is_bitsandbytes_available():
            return True
    except Exception:
        pass

    print("bitsandbytes is not available to Transformers. Installing bitsandbytes>=0.46.1 ...")
    %pip install -q -U "bitsandbytes>=0.46.1"
    importlib.invalidate_caches()

    try:
        from transformers.utils import is_bitsandbytes_available
        return is_bitsandbytes_available()
    except Exception as exc:
        print("Could not check bitsandbytes availability:", repr(exc))
        return False


if USE_LLM_STUDENT_UTTERANCES or USE_LLM_COMMUNICATION_AI or USE_LLM_TEACHER_UTTERANCES:
    if SHARED_LLM_LOAD_IN_4BIT:
        SHARED_LLM_LOAD_IN_4BIT = ensure_bitsandbytes_for_shared_llm()
        print("SHARED_LLM_LOAD_IN_4BIT:", SHARED_LLM_LOAD_IN_4BIT)

    reuse_shared_llm = (
        "shared_text_generator" in globals()
        and getattr(shared_text_generator, "model_id", None) == SHARED_LLM_MODEL_ID
        and getattr(shared_text_generator, "load_in_4bit", None) == SHARED_LLM_LOAD_IN_4BIT
    )
    if reuse_shared_llm:
        print("reuse shared LLM:", SHARED_LLM_MODEL_ID)
    else:
        print("create shared LLM:", SHARED_LLM_MODEL_ID)
        shared_text_generator = LocalLLM(
            model_id=SHARED_LLM_MODEL_ID,
            load_in_4bit=SHARED_LLM_LOAD_IN_4BIT,
            generation_config=GenerationConfig(
                max_new_tokens=256,
                temperature=0.2,
                top_p=0.9,
                do_sample=False,
                repetition_penalty=1.05,
            ),
            model_load_config=ModelLoadConfig(load_in_4bit=SHARED_LLM_LOAD_IN_4BIT),
        )
else:
    shared_text_generator = None

sim = StudentAISimulator(
    use_mock_model=USE_MOCK_MODEL,
    speech_generator=shared_text_generator if USE_LLM_STUDENT_UTTERANCES else None,
)
student_record = sim.respond(
    STUDENT_ID,
    TEACHER_MESSAGE,
    update_knowledge=False,
)
student_utterance = student_record["answer"]

print("student_speech_generator:", sim.agent.model_id)
print(student_utterance)

## 5. 伝達AIが発話を観察

伝達AIが、生徒発話から見える個人特徴を推定し、教師AIに渡す注意点を作ります。

In [ ]:
import importlib
from pprint import pprint

import src.observer.trait_classifier as trait_classifier

importlib.reload(trait_classifier)
CommunicationAI = trait_classifier.CommunicationAI
LLMCommunicationAI = trait_classifier.LLMCommunicationAI

if USE_LLM_COMMUNICATION_AI:
    communication_ai = LLMCommunicationAI(shared_text_generator)
else:
    communication_ai = CommunicationAI()

print("communication_ai:", type(communication_ai).__name__)
print("has summarize_classroom:", hasattr(communication_ai, "summarize_classroom"))
observation = communication_ai.classify_utterance(
    utterance=student_utterance,
    student_id=STUDENT_ID,
).to_dict()

pprint(observation)

## 6. 複数生徒を実際に反応させて要約

`CLASS_ID` で指定したクラスの生徒を同じ教師発話に反応させます。伝達AIには内部状態ではなく、授業中に観察できる発話、正誤、反応時間などだけを渡します。

In [ ]:
import importlib
import time
from pprint import pprint

from src.class_manager import ClassManager
from src.grader import LinearEquationGrader
import src.observer.observation_filter as observation_filter

importlib.reload(observation_filter)
build_observable_event = observation_filter.build_observable_event
events_to_communication_rows = observation_filter.events_to_communication_rows

class_manager = ClassManager()
class_definition = class_manager.load_class(CLASS_ID)
class_summary = class_manager.summarize_class(CLASS_ID)
CLASS_STUDENT_IDS = class_definition["student_ids"]

print("class_id:", CLASS_ID)
print("class_features:")
pprint(class_summary["class_features"])
print("student_count:", class_summary["student_count"])
print("average_score:", class_summary["average_score"])
print("score_std:", class_summary["score_std"])
print("low_score_students:", class_summary["low_score_students"])
print("high_score_students:", class_summary["high_score_students"])
print("trait_counts:")
pprint(class_summary["trait_counts"])

EXPECTED_ANSWER = "x = 5"
grader = LinearEquationGrader()

classroom_records = []
observable_events = []

for sid in CLASS_STUDENT_IDS:
    started = time.perf_counter()
    record = sim.respond(sid, TEACHER_MESSAGE, update_knowledge=False)
    response_time_sec = round(time.perf_counter() - started, 2)
    utterance = record["answer"]
    grade = grader.grade(EXPECTED_ANSWER, utterance)
    event = build_observable_event(
        lesson_id="L001",
        teacher_id="T001",
        student_id=sid,
        teacher_prompt=TEACHER_MESSAGE,
        utterance=utterance,
        answer=utterance,
        is_correct=grade["is_correct"],
        response_time_sec=response_time_sec,
    ).to_dict()
    classroom_records.append({"student_id": sid, "utterance": utterance, "grade": grade})
    observable_events.append(event)

classroom_rows = events_to_communication_rows(observable_events)
classroom_observation = communication_ai.summarize_classroom(classroom_rows).to_dict()

pprint({
    "classroom_records": classroom_records,
    "student_count": classroom_observation["student_count"],
    "profile_counts": classroom_observation["profile_counts"],
    "trait_level_counts": classroom_observation["trait_level_counts"],
    "priority_students": classroom_observation["priority_students"],
    "classroom_summary": classroom_observation["classroom_summary"],
    "recommended_teacher_actions": classroom_observation["recommended_teacher_actions"],
})

## 7. 複数生徒の教師側推定を更新して表で確認

各生徒の `teacher_belief` を、観察イベントと伝達AIの個別分類結果から更新します。表では推定理解度、confidence、性格・心理推定の変化を見ます。

In [ ]:
import importlib
import pandas as pd
from pprint import pprint

import src.teacher.belief_manager as belief_manager

importlib.reload(belief_manager)

TeacherBeliefManager = belief_manager.TeacherBeliefManager

belief_manager_instance = TeacherBeliefManager()
teacher_beliefs = belief_manager_instance.update_many(
    teacher_id="T001",
    observations=observable_events,
    communication_results=classroom_observation["individual_results"],
)
teacher_belief = teacher_beliefs[STUDENT_ID]

belief_rows = []
for sid, belief in teacher_beliefs.items():
    knowledge = belief["estimated_knowledge"]["linear_equation"]
    traits = belief["estimated_traits"]
    event = next(item for item in observable_events if item["student_id"] == sid)
    belief_rows.append({
        "student_id": sid,
        "is_correct": event["is_correct"],
        "response_time_sec": event["response_time_sec"],
        "estimated_score": knowledge["score"],
        "score_confidence": knowledge["confidence"],
        "self_efficacy": traits["self_efficacy"]["level"],
        "self_efficacy_conf": traits["self_efficacy"]["confidence"],
        "question_tendency": traits["question_tendency"]["level"],
        "motivation": traits["motivation"]["level"],
        "neuroticism": traits["neuroticism"]["level"],
        "evidence_count": len(belief["evidence_history"]),
    })

belief_table = pd.DataFrame(belief_rows).sort_values("student_id")
display(belief_table)
pprint(teacher_beliefs)

## 8. Optional: multi-lesson validation

This cell is expensive because it runs several lessons for all students. It is skipped by default. Set `RUN_MULTI_LESSON_VALIDATION = True` only when you want to inspect belief accumulation over multiple lessons.

In [ ]:
import importlib
import shutil
import time
from pathlib import Path

import pandas as pd

from src.grader import LinearEquationGrader
import src.observer.observation_filter as observation_filter
import src.teacher.belief_manager as belief_manager

if not RUN_MULTI_LESSON_VALIDATION:
    print("skip multi-lesson validation. Set RUN_MULTI_LESSON_VALIDATION=True in the settings cell to run it.")
    validation_dir = Path("data/teacher_beliefs_validation")
    validation_belief_manager = None
    validation_lessons = []
    validation_events_all = list(observable_events)
    validation_classroom_summaries = []
    belief_progress_rows = []
    belief_progress_table = pd.DataFrame()
else:
    import matplotlib.pyplot as plt

    importlib.reload(observation_filter)
    importlib.reload(belief_manager)

    build_observable_event = observation_filter.build_observable_event
    events_to_communication_rows = observation_filter.events_to_communication_rows
    TeacherBeliefManager = belief_manager.TeacherBeliefManager

    validation_dir = Path("data/teacher_beliefs_validation")
    if validation_dir.exists():
        shutil.rmtree(validation_dir)

    validation_belief_manager = TeacherBeliefManager(validation_dir)
    validation_grader = LinearEquationGrader()
    validation_lessons = [
        {"lesson_id": "L001", "prompt": "x + 3 = 8 \u3092\u89e3\u3044\u3066\u304f\u3060\u3055\u3044\u3002", "expected": "x = 5"},
        {"lesson_id": "L002", "prompt": "2x + 3 = 11 \u3092\u89e3\u3044\u3066\u304f\u3060\u3055\u3044\u3002", "expected": "x = 4"},
        {"lesson_id": "L003", "prompt": "3x = 15 \u3092\u89e3\u3044\u3066\u304f\u3060\u3055\u3044\u3002", "expected": "x = 5"},
    ]

    belief_progress_rows = []
    validation_events_all = []
    validation_classroom_summaries = []

    for lesson in validation_lessons:
        lesson_events = []
        for sid in CLASS_STUDENT_IDS:
            started = time.perf_counter()
            record = sim.respond(sid, lesson["prompt"], update_knowledge=False)
            response_time_sec = round(time.perf_counter() - started, 2)
            utterance = record["answer"]
            grade = validation_grader.grade(lesson["expected"], utterance)
            event = build_observable_event(
                lesson_id=lesson["lesson_id"],
                teacher_id="T001",
                student_id=sid,
                teacher_prompt=lesson["prompt"],
                utterance=utterance,
                answer=utterance,
                is_correct=grade["is_correct"],
                response_time_sec=response_time_sec,
            ).to_dict()
            lesson_events.append(event)
            validation_events_all.append(event)

        lesson_rows = events_to_communication_rows(lesson_events)
        lesson_summary = communication_ai.summarize_classroom(lesson_rows).to_dict()
        validation_classroom_summaries.append({
            "lesson_id": lesson["lesson_id"],
            "classroom_summary": lesson_summary["classroom_summary"],
            "profile_counts": lesson_summary["profile_counts"],
        })
        updated_beliefs = validation_belief_manager.update_many(
            teacher_id="T001",
            observations=lesson_events,
            communication_results=lesson_summary["individual_results"],
        )

        for sid, belief in updated_beliefs.items():
            knowledge = belief["estimated_knowledge"]["linear_equation"]
            traits = belief["estimated_traits"]
            event = next(item for item in lesson_events if item["student_id"] == sid)
            belief_progress_rows.append({
                "lesson_id": lesson["lesson_id"],
                "student_id": sid,
                "is_correct": event["is_correct"],
                "estimated_score": knowledge["score"],
                "score_confidence": knowledge["confidence"],
                "self_efficacy": traits["self_efficacy"]["level"],
                "question_tendency": traits["question_tendency"]["level"],
                "motivation": traits["motivation"]["level"],
                "neuroticism": traits["neuroticism"]["level"],
                "evidence_count": len(belief["evidence_history"]),
            })

    belief_progress_table = pd.DataFrame(belief_progress_rows).sort_values(["student_id", "lesson_id"])
    display(belief_progress_table)

    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    for sid, group in belief_progress_table.groupby("student_id"):
        axes[0].plot(group["lesson_id"], group["estimated_score"], marker="o", label=sid)
        axes[1].plot(group["lesson_id"], group["score_confidence"], marker="o", label=sid)
    axes[0].set_title("Estimated knowledge score")
    axes[0].set_ylim(0, 100)
    axes[1].set_title("Score confidence")
    axes[1].set_ylim(0, 1)
    for ax in axes:
        ax.set_xlabel("lesson")
        ax.grid(True, alpha=0.3)
        ax.legend()
    plt.tight_layout()
    plt.show()

    display(pd.DataFrame(validation_classroom_summaries))

## 9. クラス全体を見て講義全体の構成を作成

教師側beliefをクラス全体で集約し、今日の授業目標、時間配分、導入・説明・例題・演習・確認の構成を決めます。ここが研究上の中心になる講義設計レイヤーです。

In [ ]:
import importlib
from pprint import pprint

import pandas as pd
import src.teacher.context_builder as teacher_context_builder
import src.teacher.lesson_planner as lesson_planner

importlib.reload(teacher_context_builder)
importlib.reload(lesson_planner)
TeacherContextBuilder = teacher_context_builder.TeacherContextBuilder
RuleBasedLessonPlanner = lesson_planner.RuleBasedLessonPlanner

lesson_context_builder = TeacherContextBuilder()
if RUN_MULTI_LESSON_VALIDATION and validation_belief_manager is not None:
    latest_teacher_beliefs = {
        sid: validation_belief_manager.load_or_create("T001", sid)
        for sid in CLASS_STUDENT_IDS
    }
else:
    latest_teacher_beliefs = teacher_beliefs

lesson_plan = RuleBasedLessonPlanner().plan_lesson(
    teacher_beliefs=latest_teacher_beliefs,
    curriculum=lesson_context_builder.curriculum,
    total_minutes=30,
)

pprint(lesson_plan)
display(pd.DataFrame(lesson_plan["lesson_structure"]))
display(pd.DataFrame(lesson_plan["individual_support_policy"]))

## 10. Optional: scenario comparison

This cell checks whether lesson structures change under synthetic class conditions. It is skipped by default because it is not needed for the core LLM utterance experiment.

In [ ]:
import pandas as pd
from pprint import pprint

if not RUN_SCENARIO_COMPARISON:
    print("skip scenario comparison. Set RUN_SCENARIO_COMPARISON=True in the settings cell to run it.")
    lesson_plan_comparison = {}
    lesson_plan_comparison_summary = pd.DataFrame()
    lesson_plan_comparison_structure = pd.DataFrame()
else:
    def make_scenario_belief(
        score,
        self_efficacy="medium",
        question_tendency="medium",
        motivation="medium",
        neuroticism="medium",
        misconception=None,
    ):
        misconceptions = []
        if misconception:
            misconceptions.append({
                "name": misconception,
                "confidence": 0.75,
                "evidence_count": 2,
            })
        return {
            "estimated_knowledge": {
                "linear_equation": {"score": score, "confidence": 0.8},
            },
            "estimated_traits": {
                "self_efficacy": {"level": self_efficacy, "confidence": 0.7},
                "question_tendency": {"level": question_tendency, "confidence": 0.7},
                "motivation": {"level": motivation, "confidence": 0.7},
                "conscientiousness": {"level": "medium", "confidence": 0.6},
                "neuroticism": {"level": neuroticism, "confidence": 0.7},
            },
            "estimated_misconceptions": misconceptions,
        }


    lesson_plan_scenarios = {
        "LOW_UNDERSTANDING_CLASS": {
            "S001": make_scenario_belief(25, self_efficacy="low", question_tendency="low", motivation="medium"),
            "S002": make_scenario_belief(35, self_efficacy="low", question_tendency="medium", neuroticism="high"),
            "S003": make_scenario_belief(42, self_efficacy="medium", question_tendency="low", motivation="low"),
        },
        "HIGH_UNDERSTANDING_CLASS": {
            "S001": make_scenario_belief(72, self_efficacy="high", question_tendency="medium", motivation="high"),
            "S002": make_scenario_belief(78, self_efficacy="medium", question_tendency="low", motivation="high"),
            "S003": make_scenario_belief(85, self_efficacy="high", question_tendency="medium", motivation="high"),
        },
        "MIXED_UNDERSTANDING_CLASS": {
            "S001": make_scenario_belief(35, self_efficacy="low", question_tendency="high", neuroticism="high"),
            "S002": make_scenario_belief(62, self_efficacy="medium", question_tendency="medium"),
            "S003": make_scenario_belief(88, self_efficacy="high", question_tendency="low", motivation="high"),
        },
        "LOW_QUESTION_TENDENCY_CLASS": {
            "S001": make_scenario_belief(62, question_tendency="low"),
            "S002": make_scenario_belief(68, question_tendency="low"),
            "S003": make_scenario_belief(74, question_tendency="low"),
        },
        "COEFFICIENT_MISCONCEPTION_CLASS": {
            "S001": make_scenario_belief(58, misconception="coefficient misconception likely"),
            "S002": make_scenario_belief(63, misconception="subtracting 3 in 3x = 15"),
            "S003": make_scenario_belief(70),
        },
    }

    def misconception_names(items):
        return [str(item.get("name", item)) if isinstance(item, dict) else str(item) for item in items]


    lesson_plan_comparison = {}
    lesson_plan_summary_rows = []
    lesson_plan_structure_rows = []

    planner = RuleBasedLessonPlanner()
    for scenario_name, scenario_beliefs in lesson_plan_scenarios.items():
        scenario_plan = planner.plan_lesson(
            teacher_beliefs=scenario_beliefs,
            curriculum=lesson_context_builder.curriculum,
            total_minutes=30,
        )
        lesson_plan_comparison[scenario_name] = scenario_plan
        profile = scenario_plan["class_profile"]
        lesson_plan_summary_rows.append({
            "scenario": scenario_name,
            "target_skill": scenario_plan["lesson_goal"]["target_skill"],
            "goal_text": scenario_plan["lesson_goal"]["goal_text"],
            "average_estimated_score": profile["average_estimated_score"],
            "score_std": profile["score_std"],
            "low_score_students": ",".join(profile["low_score_students"]),
            "high_score_students": ",".join(profile["high_score_students"]),
            "common_misconceptions": " / ".join(misconception_names(profile["common_misconceptions"])),
            "common_risks": " / ".join(profile["common_risks"]),
        })
        for phase in scenario_plan["lesson_structure"]:
            lesson_plan_structure_rows.append({"scenario": scenario_name, **phase})

    lesson_plan_comparison_summary = pd.DataFrame(lesson_plan_summary_rows)
    lesson_plan_comparison_structure = pd.DataFrame(lesson_plan_structure_rows)

    display(lesson_plan_comparison_summary)
    display(lesson_plan_comparison_structure)
    pprint(lesson_plan_comparison)

## 11. クラス全体対応と個別対応を計画

講義全体の構成を踏まえ、直近の観察イベントから、次の局面での全体向け授業行動と個別支援を分けて計画します。

In [ ]:
import importlib
from pprint import pprint

import pandas as pd
import src.teacher.intervention_planner as intervention_planner

importlib.reload(intervention_planner)
RuleBasedInterventionPlanner = intervention_planner.RuleBasedInterventionPlanner

if RUN_MULTI_LESSON_VALIDATION and validation_lessons:
    latest_lesson_id = validation_lessons[-1]["lesson_id"]
    latest_events = [event for event in validation_events_all if event["lesson_id"] == latest_lesson_id]
    latest_classroom_observation = communication_ai.summarize_classroom(
        events_to_communication_rows(latest_events)
    ).to_dict()
else:
    latest_lesson_id = "CORE_OBSERVATION"
    latest_events = observable_events
    latest_classroom_observation = classroom_observation

intervention_plan = RuleBasedInterventionPlanner().plan(
    classroom_observation=latest_classroom_observation,
    teacher_beliefs=latest_teacher_beliefs,
    lesson_goal=lesson_plan["lesson_goal"],
    recent_events=latest_events,
    next_problem_bank=lesson_context_builder.curriculum["next_problem_bank"],
)

pprint(intervention_plan)
display(pd.DataFrame(intervention_plan["individual_supports"]))

## 12. Convert intervention plan into teacher utterances

If `USE_LLM_TEACHER_UTTERANCES = True`, this cell uses the shared LLM. Otherwise it uses the rule-based fallback.

In [ ]:
import importlib
from pprint import pprint

import src.teacher.utterance_builder as utterance_builder

importlib.reload(utterance_builder)
RuleBasedTeacherUtteranceBuilder = utterance_builder.RuleBasedTeacherUtteranceBuilder
LLMTeacherUtteranceBuilder = utterance_builder.LLMTeacherUtteranceBuilder

if USE_LLM_TEACHER_UTTERANCES:
    teacher_utterance_builder = LLMTeacherUtteranceBuilder(shared_text_generator)
else:
    teacher_utterance_builder = RuleBasedTeacherUtteranceBuilder()

teacher_utterance_plan = teacher_utterance_builder.build(intervention_plan)

print("teacher_utterance_builder:", type(teacher_utterance_builder).__name__)
print("generation_mode:", teacher_utterance_plan.get("generation_mode", "rule_based"))
print("whole_class_utterance:")
print(teacher_utterance_plan["whole_class_utterance"])
print("\nindividual_utterances:")
for item in teacher_utterance_plan["individual_utterances"]:
    print(f"- {item['student_id']} ({item['support_type']}): {item['utterance']}")

pprint(teacher_utterance_plan)

## 13. Optional: run the next turn with generated teacher utterance

This cell sends the generated teacher utterance back to all students. It is skipped by default to avoid extra LLM calls.

In [ ]:
import time
from pprint import pprint

import pandas as pd
from src.grader import LinearEquationGrader

if not RUN_NEXT_TURN_SIMULATION:
    print("skip next-turn simulation. Set RUN_NEXT_TURN_SIMULATION=True in the settings cell to run it.")
    next_turn_records = []
    next_turn_events = []
    next_turn_classroom_observation = {}
    next_turn_beliefs = {}
    next_turn_belief_table = pd.DataFrame()
else:
    next_teacher_message = teacher_utterance_plan["whole_class_utterance"]
    next_expected_answer = teacher_utterance_plan.get("expected_answer")
    next_turn_lesson_id = "L004"
    next_turn_grader = LinearEquationGrader()

    next_turn_records = []
    next_turn_events = []

    print("next turn teacher message:")
    print(next_teacher_message)
    print("
individual support candidates:")
    for item in teacher_utterance_plan["individual_utterances"]:
        print(f"- {item['student_id']}: {item['utterance']}")

    for sid in CLASS_STUDENT_IDS:
        started = time.perf_counter()
        record = sim.respond(sid, next_teacher_message, update_knowledge=False)
        response_time_sec = round(time.perf_counter() - started, 2)
        utterance = record["answer"]
        grade = (
            next_turn_grader.grade(next_expected_answer, utterance)
            if next_expected_answer
            else {"is_correct": None, "score": None, "expected_value": None, "student_value": None}
        )
        event = build_observable_event(
            lesson_id=next_turn_lesson_id,
            teacher_id="T001",
            student_id=sid,
            teacher_prompt=next_teacher_message,
            utterance=utterance,
            answer=utterance,
            is_correct=grade["is_correct"],
            response_time_sec=response_time_sec,
        ).to_dict()
        next_turn_records.append({"student_id": sid, "utterance": utterance, "grade": grade})
        next_turn_events.append(event)

    next_turn_classroom_observation = communication_ai.summarize_classroom(
        events_to_communication_rows(next_turn_events)
    ).to_dict()
    belief_manager_for_next_turn = validation_belief_manager or belief_manager_instance
    next_turn_beliefs = belief_manager_for_next_turn.update_many(
        teacher_id="T001",
        observations=next_turn_events,
        communication_results=next_turn_classroom_observation["individual_results"],
    )

    next_turn_rows = []
    for sid, belief in next_turn_beliefs.items():
        knowledge = belief["estimated_knowledge"]["linear_equation"]
        traits = belief["estimated_traits"]
        event = next(item for item in next_turn_events if item["student_id"] == sid)
        next_turn_rows.append({
            "student_id": sid,
            "is_correct": event["is_correct"],
            "estimated_score": knowledge["score"],
            "score_confidence": knowledge["confidence"],
            "self_efficacy": traits["self_efficacy"]["level"],
            "question_tendency": traits["question_tendency"]["level"],
            "motivation": traits["motivation"]["level"],
            "neuroticism": traits["neuroticism"]["level"],
            "evidence_count": len(belief["evidence_history"]),
        })

    next_turn_belief_table = pd.DataFrame(next_turn_rows).sort_values("student_id")
    display(next_turn_belief_table)
    pprint({
        "next_turn_records": next_turn_records,
        "next_turn_classroom_summary": next_turn_classroom_observation["classroom_summary"],
        "next_turn_profile_counts": next_turn_classroom_observation["profile_counts"],
    })

## 14. Optional: run a full multi-turn lesson session

This cell runs the entire lesson plan across phases. It is skipped by default because it is the slowest validation step.

In [ ]:
import importlib
import shutil
from pathlib import Path
from pprint import pprint

import pandas as pd
import src.teacher.belief_manager as session_belief_manager_module
import src.teacher.lesson_session_runner as lesson_session_runner_module

if not RUN_AUTO_LESSON_SESSION:
    print("skip auto lesson session. Set RUN_AUTO_LESSON_SESSION=True in the settings cell to run it.")
    lesson_session_result = {}
    lesson_session_table = pd.DataFrame()
else:
    importlib.reload(session_belief_manager_module)
    importlib.reload(lesson_session_runner_module)
    TeacherBeliefManager = session_belief_manager_module.TeacherBeliefManager
    LessonSessionRunner = lesson_session_runner_module.LessonSessionRunner

    session_belief_dir = Path("data/teacher_beliefs_session_validation")
    if session_belief_dir.exists():
        shutil.rmtree(session_belief_dir)

    session_runner = LessonSessionRunner(
        student_simulator=sim,
        communication_ai=communication_ai,
        belief_manager=TeacherBeliefManager(session_belief_dir),
        teacher_id="T001",
        update_student_knowledge=False,
    )

    lesson_session_result = session_runner.run_lesson(
        lesson_id="LESSON_AUTO_001",
        student_ids=CLASS_STUDENT_IDS,
        lesson_plan=lesson_plan,
        curriculum=lesson_context_builder.curriculum,
        initial_teacher_beliefs=latest_teacher_beliefs,
    )

    lesson_session_rows = []
    for turn in lesson_session_result["turns"]:
        graded_events = [event for event in turn["events"] if event["is_correct"] is not None]
        correct_count = sum(1 for event in graded_events if event["is_correct"] is True)
        individual_message_count = sum(
            1
            for event in turn["events"]
            if event["teacher_prompt"] != turn["teacher_message"]
        )
        lesson_session_rows.append({
            "phase_index": turn["phase_index"],
            "phase": turn["phase"]["phase"],
            "minutes": turn["phase"].get("minutes"),
            "student_events": len(turn["events"]),
            "expected_answer": turn["expected_answer"],
            "correct_count": correct_count,
            "accuracy": round(correct_count / len(graded_events), 3) if graded_events else None,
            "individual_message_count": individual_message_count,
        })

    lesson_session_table = pd.DataFrame(lesson_session_rows)
    display(lesson_session_table)
    pprint(lesson_session_result["summary"])
    pprint(lesson_session_result["final_class_profile"])

## 15. Optional: legacy single-student teacher context

This older single-student context path is kept for comparison, but it is skipped by default. The current research core uses class-level observation and lesson planning.

In [ ]:
import importlib
from pprint import pprint

from src.state_manager import StateManager
import src.teacher.context_builder as teacher_context_builder

if not RUN_LEGACY_SINGLE_STUDENT_CONTEXT:
    print("skip legacy single-student teacher context. Set RUN_LEGACY_SINGLE_STUDENT_CONTEXT=True in the settings cell to run it.")
    teacher_context = {}
else:
    importlib.reload(teacher_context_builder)
    TeacherContextBuilder = teacher_context_builder.TeacherContextBuilder

    state_manager = StateManager()
    student_state = state_manager.load_student(STUDENT_ID)

    context_builder = TeacherContextBuilder()
    teacher_context = context_builder.build_context(
        student_state=student_state,
        recent_student_utterance=student_utterance,
        communication_observation=observation,
        classroom_observation=classroom_observation,
        teacher_belief=teacher_belief,
    )

    pprint({
        "target_skill": teacher_context["target_skill"],
        "lesson_goal": teacher_context["lesson_goal"],
        "student_state_summary": teacher_context["student_state_summary"],
        "communication_ai_observation": teacher_context["communication_ai_observation"],
    })

## 16. Optional: legacy single-student strategy selection

This older strategy selector is skipped by default. Use it only when comparing the old individual-focused path with the class-level lesson design path.

In [ ]:
from pprint import pprint

from src.teacher import RuleBasedTeachingStrategySelector

if not RUN_LEGACY_SINGLE_STUDENT_CONTEXT:
    print("skip legacy single-student strategy selection.")
    teacher_decision = {}
else:
    selector = RuleBasedTeachingStrategySelector()
    teacher_decision = selector.select_strategy(teacher_context)
    pprint(teacher_decision)

## 17. 最新結果を保存

あとで比較できるように、教師AI用コンテキストと授業方略の判断結果をJSONで保存します。

In [ ]:
import json
from pathlib import Path

output_dir = Path("data/assessments")
output_dir.mkdir(parents=True, exist_ok=True)


def has_var(name):
    return name in globals()


def save_json_if_available(var_name, filename):
    if not has_var(var_name):
        return None
    path = output_dir / filename
    path.write_text(json.dumps(globals()[var_name], ensure_ascii=False, indent=2, default=str), encoding="utf-8")
    return path


def save_csv_if_available(var_name, filename):
    value = globals().get(var_name)
    if value is None or not hasattr(value, "to_csv"):
        return None
    path = output_dir / filename
    value.to_csv(path, index=False)
    return path

saved_paths = []
for item in [
    save_json_if_available("class_summary", "class_summary_latest.json"),
    save_json_if_available("teacher_context", "teacher_context_latest.json"),
    save_json_if_available("teacher_decision", "teaching_strategy_decision_latest.json"),
    save_json_if_available("observable_events", "observable_events_latest.json"),
    save_json_if_available("teacher_beliefs", "teacher_beliefs_latest.json"),
    save_csv_if_available("belief_table", "teacher_belief_table_latest.csv"),
    save_csv_if_available("belief_progress_table", "teacher_belief_progress_validation.csv"),
    save_json_if_available("validation_events_all", "observable_events_validation.json"),
    save_json_if_available("lesson_plan", "lesson_plan_latest.json"),
    save_json_if_available("lesson_plan_comparison", "lesson_plan_comparison.json"),
    save_csv_if_available("lesson_plan_comparison_summary", "lesson_plan_comparison_summary.csv"),
    save_csv_if_available("lesson_plan_comparison_structure", "lesson_plan_comparison_structure.csv"),
    save_json_if_available("lesson_session_result", "lesson_session_result_latest.json"),
    save_csv_if_available("lesson_session_table", "lesson_session_table_latest.csv"),
    save_json_if_available("intervention_plan", "intervention_plan_latest.json"),
    save_json_if_available("teacher_utterance_plan", "teacher_utterance_plan_latest.json"),
    save_json_if_available("next_turn_events", "next_turn_observable_events_latest.json"),
    save_json_if_available("next_turn_beliefs", "next_turn_teacher_beliefs_latest.json"),
    save_csv_if_available("next_turn_belief_table", "next_turn_belief_table_latest.csv"),
]:
    if item is not None:
        saved_paths.append(item)

if SAVE_FULL_ARTIFACTS:
    print("saved files:")
    for path in saved_paths:
        print("saved:", path)
else:
    print("saved core/available artifacts:", len(saved_paths))
    print("Set SAVE_FULL_ARTIFACTS=True to print every saved path.")

## 共有用結果サマリーを1ファイルに保存

ここまでの主要な結果を `data/assessments/teaching_strategy_result_summary.md` にまとめます。

In [ ]:
from datetime import datetime
import json
from pathlib import Path

output_dir = Path("data/assessments")
output_dir.mkdir(parents=True, exist_ok=True)
result_summary_path = output_dir / "teaching_strategy_result_summary.md"


def has_var(name):
    return name in globals()


def get_var(name, default=None):
    return globals().get(name, default)


def to_json_text(value):
    return json.dumps(value, ensure_ascii=False, indent=2, default=str)


def add_json_section(lines, title, var_name):
    lines.append(f"## {title}")
    if has_var(var_name):
        lines.append("```json")
        lines.append(to_json_text(get_var(var_name)))
        lines.append("```")
    else:
        lines.append(f"`{var_name}` is not available. Run the related cell first.")
    lines.append("")


def add_text_section(lines, title, value):
    lines.append(f"## {title}")
    lines.append(str(value) if value is not None else "not available")
    lines.append("")


def add_table_section(lines, title, var_name, max_rows=20):
    lines.append(f"## {title}")
    value = get_var(var_name)
    if value is None:
        lines.append(f"`{var_name}` is not available. Run the related cell first.")
    elif hasattr(value, "head") and hasattr(value, "to_markdown"):
        lines.append(value.head(max_rows).to_markdown(index=False))
    else:
        lines.append("```json")
        lines.append(to_json_text(value))
        lines.append("```")
    lines.append("")


summary_lines = []
summary_lines.append("# Teaching Strategy Experiment Result Summary")
summary_lines.append("")
summary_lines.append(f"Generated at: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
summary_lines.append("")
summary_lines.append("## Experiment Settings")
summary_lines.append(f"- student_id: {get_var('STUDENT_ID', 'not available')}")
summary_lines.append(f"- class_id: {get_var('CLASS_ID', 'not available')}")
summary_lines.append(f"- use_llm_student_utterances: {get_var('USE_LLM_STUDENT_UTTERANCES', 'not available')}")
summary_lines.append(f"- use_llm_communication_ai: {get_var('USE_LLM_COMMUNICATION_AI', 'not available')}")
summary_lines.append(f"- use_llm_teacher_utterances: {get_var('USE_LLM_TEACHER_UTTERANCES', 'not available')}")
summary_lines.append(f"- shared_llm_model_id: {get_var('SHARED_LLM_MODEL_ID', 'not available')}")
summary_lines.append(f"- shared_llm_load_in_4bit: {get_var('SHARED_LLM_LOAD_IN_4BIT', 'not available')}")
summary_lines.append("")

add_text_section(summary_lines, "Teacher Input", get_var("TEACHER_MESSAGE", "not available"))
add_text_section(summary_lines, "Student AI Utterance", get_var("student_utterance", "not available"))
add_json_section(summary_lines, "Communication AI Individual Estimate", "observation")
add_json_section(summary_lines, "Communication AI Classroom Summary", "classroom_observation")
add_json_section(summary_lines, "Teacher AI Lesson Plan", "lesson_plan")
add_json_section(summary_lines, "Teacher AI Intervention Plan", "intervention_plan")
add_json_section(summary_lines, "Teacher AI Utterance Result", "teacher_utterance_plan")
add_table_section(summary_lines, "Teacher Belief Table", "belief_table")
add_table_section(summary_lines, "Teacher Belief Table After Next Turn", "next_turn_belief_table")
add_json_section(summary_lines, "Lesson Session Result", "lesson_session_result")

result_summary_path.write_text("\n".join(summary_lines), encoding="utf-8")
print("saved result summary:", result_summary_path)